## 21_spatial_residuals
  Moran's I (queen contiguity, 9999 permutations) of the scaling residuals (2023) and of the
  2021 commuting-block residuals. Output: results/spatial_residuals/*.csv.

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

# ---- Config (change names and paths here only) ----
ROOT_MARKER = Path("src") / "config.py"      # file that exists only in the project root
MASTER_PATH = Path("data/processed/nuts3_master_table.parquet")
OUT_DIR = Path("results/spatial_residuals")
RESID_CSV = OUT_DIR / "residuals_2023.csv"
YEAR = 2023
COL_ID, COL_YEAR, COL_COUNTRY = "nuts3_id", "year", "country_code"
COL_N, COL_E, COL_Y = "population", "employment", "gdp_total"
MODELS = {"beta_Y": "ln_y", "beta_E": "ln_e", "beta_P": "ln_p"}   # label -> dependent variable
# ----------------------------------------------------

# Project root: walk up from the working folder until ROOT_MARKER is found
start = Path.cwd()
candidates = [start, *start.parents]
root = next((p for p in candidates if (p / ROOT_MARKER).is_file()), None)
if root is None:
    raise FileNotFoundError(
        f"Nem található a {ROOT_MARKER.as_posix()} fájl a munkakönyvtár egyik "
        "szülőmappájában sem - indítsd a notebookot a projekten belülről."
    )
os.chdir(root)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same sample as notebook 19: year filter, all three variables positive
df = pd.read_parquet(MASTER_PATH, columns=[COL_ID, COL_YEAR, COL_COUNTRY, COL_N, COL_E, COL_Y])
df = df[df[COL_YEAR] == YEAR].copy()
mask = (df[[COL_N, COL_E, COL_Y]].notna().all(axis=1)
        & (df[[COL_N, COL_E, COL_Y]] > 0).all(axis=1))
df = df[mask].reset_index(drop=True)

df["ln_n"] = np.log(df[COL_N])
df["ln_y"] = np.log(df[COL_Y])
df["ln_e"] = np.log(df[COL_E])
df["ln_p"] = df["ln_y"] - df["ln_e"]

# Residuals of the three models (ln N + country fixed effects)
resid = df[[COL_ID, COL_COUNTRY, "ln_n"]].copy()
for label, dep in MODELS.items():
    res = smf.ols(f"{dep} ~ ln_n + C({COL_COUNTRY})", data=df).fit()
    resid[f"resid_{label}"] = res.resid
    print(f"{label}: slope {res.params['ln_n']:.4f}, n = {int(res.nobs)}")

# Identity check: resid_Y = resid_E + resid_P (floating-point level)
id_diff = (resid["resid_beta_Y"] - resid["resid_beta_E"] - resid["resid_beta_P"]).abs().max()
print(f"Max |resid_Y - resid_E - resid_P| = {id_diff:.2e}")
print("Regions per country:", resid[COL_COUNTRY].value_counts().to_dict())

resid.to_csv(RESID_CSV, index=False)
print("Saved:", RESID_CSV.as_posix())

In [ ]:
import warnings
import geopandas as gpd
import shapely.wkb
from geopandas.array import GeometryArray
from libpysal import weights
from esda.moran import Moran

# ----------------------------- CONFIG (Moran) -----------------------------
GEOM_COL = "geometry"                 # WKB geometry column of the master table
CRS_GEO = "EPSG:4326"
W_TRANSFORM = "r"                     # row-standardised weights
N_PERM = 9999                         # permutations for the pseudo p-value
SEED = 0
MORAN_CSV = OUT_DIR / "moran_2023.csv"
# --------------------------------------------------------------------------

# Geometries of the sample regions (geometry stored as WKB bytes)
geo = pd.read_parquet(MASTER_PATH, columns=[COL_ID, COL_YEAR, GEOM_COL])
geo = geo[geo[COL_YEAR] == YEAR].set_index(COL_ID)
geoms = geo[GEOM_COL].apply(
    lambda x: shapely.wkb.loads(x) if isinstance(x, (bytes, bytearray)) else x
).tolist()
geom_arr = np.empty(len(geoms), dtype=object)
for i, g in enumerate(geoms):
    geom_arr[i] = g
geom_by_id = gpd.GeoSeries(GeometryArray(geom_arr, crs=CRS_GEO), index=geo.index)

# GeoDataFrame in the same row order as the residual table
gdf = gpd.GeoDataFrame(resid.copy(), geometry=geom_by_id.reindex(resid[COL_ID]).to_numpy(),
                       crs=CRS_GEO).reset_index(drop=True)
assert gdf.geometry.notna().all(), "a region has no geometry"

# Queen contiguity weights; warnings are recorded by class name only
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    w = weights.Queen.from_dataframe(gdf)
print("Weights warnings:", ";".join(sorted({x.category.__name__ for x in caught})) or "none")
print("Regions:", w.n, "| islands (no neighbour):", len(w.islands),
      "| connected components:", w.n_components,
      "| mean neighbours:", round(w.mean_neighbors, 2))
if w.islands:
    print("Island ids:", [gdf[COL_ID].iloc[i] for i in w.islands])

# Moran's I for each residual column
w.transform = W_TRANSFORM
np.random.seed(SEED)
rows = []
for label in MODELS:
    y = gdf[f"resid_{label}"].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mi = Moran(y, w, permutations=N_PERM)
    rows.append({"model": label, "n": w.n, "moran_I": mi.I, "expected_I": mi.EI,
                 "z_sim": mi.z_sim, "p_sim": mi.p_sim, "permutations": N_PERM,
                 "weights": "queen", "n_islands": len(w.islands),
                 "n_components": w.n_components})

moran = pd.DataFrame(rows)
moran.to_csv(MORAN_CSV, index=False)
print(moran.round(4).to_string(index=False))
print("Saved:", MORAN_CSV.as_posix())

In [ ]:
# ----------------------------- CONFIG (subsets) -----------------------------
CAPITAL_IDS = ["CZ010", "PL911", "HU110", "SK010"]   # Praha, Warszawa, Budapest, Bratislava
SUBSETS_CSV = OUT_DIR / "moran_2023_subsets.csv"
# ----------------------------------------------------------------------------

# The capital ids must exist in the sample, otherwise the "without capitals" run is wrong
missing_caps = [c for c in CAPITAL_IDS if c not in set(gdf[COL_ID])]
assert not missing_caps, f"capital id not found in the sample: {missing_caps}"

# Subsets: name -> boolean mask over gdf
subsets = {"all": np.ones(len(gdf), dtype=bool)}
for country in sorted(gdf[COL_COUNTRY].unique()):
    subsets[country] = (gdf[COL_COUNTRY] == country).to_numpy()
subsets["without_capitals"] = ~gdf[COL_ID].isin(CAPITAL_IDS).to_numpy()


def moran_for(sub_gdf, label):
    """Moran's I of every residual column on a subset, with weights rebuilt on the subset."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        w_sub = weights.Queen.from_dataframe(sub_gdf)
    w_sub.transform = W_TRANSFORM
    out = []
    for model in MODELS:
        np.random.seed(SEED)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mi = Moran(sub_gdf[f"resid_{model}"].to_numpy(), w_sub, permutations=N_PERM)
        out.append({"subset": label, "model": model, "n": w_sub.n, "moran_I": mi.I,
                    "expected_I": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim,
                    "permutations": N_PERM, "n_islands": len(w_sub.islands),
                    "n_components": w_sub.n_components})
    return out


rows = []
for label, mask in subsets.items():
    sub = gdf[mask].reset_index(drop=True)
    rows.extend(moran_for(sub, label))

moran_subsets = pd.DataFrame(rows)
moran_subsets.to_csv(SUBSETS_CSV, index=False)
print(moran_subsets.round(4).to_string(index=False))
print("Saved:", SUBSETS_CSV.as_posix())

In [ ]:
from pathlib import Path

# ----------------------------- CONFIG (2021 eta) -----------------------------
EXCESS_CSV = Path("results/core_scaling_decomposition/excess_attraction_2021.csv")
GAMMA_CSV = Path("results/core_scaling_decomposition/excess_attraction_regression_2021.csv")
COL_ETA = "eta_Y"
A_COLS = ["A_gravity_power", "A_gravity_exp", "A_radiation"]   # network variables of notebook 19
COV_TYPE = "HC3"
ETA_MORAN_CSV = OUT_DIR / "moran_2021_eta.csv"
# -----------------------------------------------------------------------------

ex = pd.read_csv(EXCESS_CSV)
ex = ex.dropna(subset=[COL_ETA, *A_COLS]).reset_index(drop=True)

# Geometries by region id (2023 table, same NUTS3 ids); every id must be found
geom_2021 = geom_by_id.reindex(ex[COL_ID])
assert geom_2021.notna().all(), "a 2021 region has no geometry in the 2023 table"
gdf21 = gpd.GeoDataFrame(ex.copy(), geometry=geom_2021.to_numpy(), crs=CRS_GEO)

# Residual of the gamma regression for each network variable (same spec as notebook 19)
gamma_rows = []
for a in A_COLS:
    res = smf.ols(f"{COL_ETA} ~ {a} + C({COL_COUNTRY})", data=gdf21).fit(cov_type=COV_TYPE)
    gdf21[f"resid_{a}"] = res.resid
    gamma_rows.append({"regressor": a, "gamma": res.params[a], "se": res.bse[a], "n": int(res.nobs)})
print("Re-run gamma (compare with the saved file below):")
print(pd.DataFrame(gamma_rows).round(4).to_string(index=False))
print("Saved gamma table:")
print(pd.read_csv(GAMMA_CSV).round(4).to_string(index=False))

# Variables to test: eta itself and the three regression residuals
TEST_COLS = [COL_ETA] + [f"resid_{a}" for a in A_COLS]

capitals_in_sample = [c for c in CAPITAL_IDS if c in set(gdf21[COL_ID])]
subsets21 = {"all": np.ones(len(gdf21), dtype=bool)}
for country in sorted(gdf21[COL_COUNTRY].unique()):
    subsets21[country] = (gdf21[COL_COUNTRY] == country).to_numpy()
subsets21["without_capitals"] = ~gdf21[COL_ID].isin(capitals_in_sample).to_numpy()

rows = []
for label, mask in subsets21.items():
    sub = gdf21[mask].reset_index(drop=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        w_sub = weights.Queen.from_dataframe(sub)
    w_sub.transform = W_TRANSFORM
    for col in TEST_COLS:
        np.random.seed(SEED)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            mi = Moran(sub[col].to_numpy(), w_sub, permutations=N_PERM)
        rows.append({"subset": label, "variable": col, "n": w_sub.n, "moran_I": mi.I,
                     "expected_I": mi.EI, "z_sim": mi.z_sim, "p_sim": mi.p_sim,
                     "permutations": N_PERM, "n_islands": len(w_sub.islands),
                     "n_components": w_sub.n_components})

moran_eta = pd.DataFrame(rows)
moran_eta.to_csv(ETA_MORAN_CSV, index=False)
print("Capitals removed in the last subset:", capitals_in_sample)
print(moran_eta.round(4).to_string(index=False))
print("Saved:", ETA_MORAN_CSV.as_posix())